In [1]:
!pip install ml4eft
!pip install wget
!lhapdf install NNPDF31_lo_as_0118

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Obtaining dependency information for torch from https://files.pythonhosted.org/packages/d1/bd/9912d30b68845256aabbb4a40aeefeef3c3b20db5211ccda653544ada4b6/torch-2.11.0-cp311-cp311-win_amd64.whl.metadata
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Obtaining dependency information for pylhe>=0.4.0 from https://files.pythonhosted.org/packages/81/fa/60af542e82803216e2ec89ad5cc6bf73c35e0afcf07af24a530f7e94651d/pylhe-1.0.2-py3-none-any.whl.metadata
  Obtaining dependency information for awkward>=1.2.0 from https://files.pythonhosted.org/packages/77/39/4d8414260c3d83f22029a39e51553c173611b378d62ca391e5ca68e65cfa/awkward-2.9.0-py3-none-any.whl.metadata
  Obtaining dependency information for graphviz>=0.19 from https://files.pythonhosted.org/packages/91/4c/e0ce1ef95d4000ebc1c11801f9b944fa5910ecc15b5e351865763d8657f8/graphviz-0.21-py3

'lhapdf' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
import wget
import tarfile
import os
import time
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
from matplotlib import rc
import sys
import json
import numpy.core.numeric as numeric
#sys.modules["numpy._core.numeric"] = numeric

import ml4eft.core.classifier as classifier
import ml4eft.analyse.analyse as analyse
from ml4eft.analyse.animate import Animate
import ml4eft.plotting.features as features
import ml4eft.preproc.constants as constants

import types
sys.modules['pymultinest'] = types.ModuleType('pymultinest')
import ml4eft.limits.optimize_ns as optimize

mt = constants.mt

rc('text', usetex=False)
rc('font', **{'family': 'DejaVu Sans', 'size': 22})
plt.rcParams.update({'font.size': 12})

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "c:\Users\jtpit\anaconda3\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [ ]:
# Files that the ML trains on, will load these in from the path given in the json not from these variable names directly
# For the ML to find the fill, must follow format for quad: {path to dir}/tt_{WC name 1}_{WC name 2}/events_0.pkl
# df_sm is used for theory calculations

df_sm = pd.read_pickle("/depot/cms/top/jpittard/Purdue_Analysis_EFT/pickel_files/sm_quad_pickles/tt_sm/events_0.pkl.gz")
df_ctgre = pd.read_pickle("/depot/cms/top/jpittard/Purdue_Analysis_EFT/pickel_files/sm_quad_pickles/tt_ctGRe_ctGRe/events_0.pkl.gz")
df_ctu8 = pd.read_pickle("/depot/cms/top/jpittard/Purdue_Analysis_EFT/pickel_files/sm_quad_pickles/tt_ctu8_ctu8/events_0.pkl.gz")
df_ctgre_ctu8 = pd.read_pickle("/depot/cms/top/jpittard/Purdue_Analysis_EFT/pickel_files/sm_quad_pickles/tt_ctGRe_ctu8/events_0.pkl.gz")
df_ctu8_ctGRe = pd.read_pickle("/depot/cms/top/jpittard/Purdue_Analysis_EFT/pickel_files/sm_quad_pickles/tt_ctu8_ctGRe/events_0.pkl.gz")

df_sm = df_sm.iloc[1:, :]
df_ctgre = df_ctgre.iloc[1:,:]
df_ctu8 = df_ctu8.iloc[1:,:]
df_ctgre_ctu8 = df_ctgre_ctu8.iloc[1:,:]
df_ctu8_ctGRe = df_ctu8_ctGRe.iloc[1:,:]

df_sm

In [ ]:
#Plots feature vectors for files loaded in

sub_features_dict = {
    #'gen_ttbar_pt': r'$p_T^{t\bar{t}}\;[\mathrm{GeV}]$',
    #'gen_ttbar_phi': r'$\phi_{t\bar{t}}$',
    #'gen_ttbar_rapidity': r'$y_{t\bar{t}}$',
    #'gen_ttbar_eta': r'$\eta_{t\bar{t}}$',
    #'gen_ttbar_delta_phi': r'$\Delta\phi(t,\bar{t})$',
    #'gen_ttbar_delta_eta': r'$\Delta\eta(t,\bar{t})$',
    'gen_ttbar_rapidity': r'$\Delta y(t,\bar{t})$',
    'gen_ttbar_mass': r'$m_{t\bar{t}}\;[\mathrm{GeV}]$',
    #'gen_llbar_delta_phi': r'$\Delta\phi(\ell,\bar{\ell})$'
}
legend_labels = [r'$c_{tu8}$', r'$c_{tGRe}$',r'$c_{tGRe,tu8}$',r'$c_{tu8,tGRe}$']

df_eft_list = [df_ctgre,df_ctu8,df_ctgre_ctu8,df_ctu8_ctGRe]

fig = features.plot_features(df_sm, df_eft_list, sub_features_dict, legend_labels);

In [ ]:
# Loads in ML4EFT json runcard for the ML
def file_downloader(url, download_dir='./downloads'):
    if not os.path.exists(download_dir):
        os.mkdir(download_dir)
    file = wget.download(url, out=download_dir)
    return file

path_to_runcard = 'https://dl.dropboxusercontent.com/s/v4ulo6icveh63fw/run_card_tt_llvlvlbb.json?dl=0'
runcard = file_downloader(path_to_runcard)

In [ ]:
#Changing the loss function that ML4EFT uses to a weighted loss

def new_loss_fn(self, outputs, labels, w_e):
    import torch

    eps = 1e-7
    outputs = torch.clamp(outputs, eps, 1 - eps)

    if self.loss_type == 'CE':

        loss = - (1 - labels) * w_e * torch.log(1 - outputs) \
               - labels * w_e * torch.log(outputs)

    elif self.loss_type == 'QC':

        loss = (1 - labels) * w_e * outputs ** 2 \
             + labels * w_e * (1 - outputs) ** 2

    return torch.sum(loss) / torch.sum(w_e)

classifier.Fitter.loss_fn = new_loss_fn

In [ ]:
# Selecting what features to train the ML on
# ML4EFT CAN ONLY CALCULATE THE DIFFERENTIAL CROSS-SECTION FOR MTTBAR AND Y !!!!
# Must be features avalible in the files used

ML_features = [
#'gen_ttbar_pt',
#'gen_ttbar_phi',
'gen_ttbar_rapidity',
#'gen_ttbar_eta',
#'gen_ttbar_delta_phi',
#'gen_ttbar_delta_eta',
#'gen_ttbar_delta_rapidity',
'gen_ttbar_mass',
#'gen_llbar_pt',
#'gen_l_eta',
]

In [ ]:
# Changing parameters of the json file
# 'features' is the previously defined features the ML will train on
# 'event_data' is the files that will be used to train the ML
# 'c_train' is the WC inputs and the corresponding value

with open(runcard) as json_runcard:
    json_runcard_loaded = json.load(json_runcard)
    
json_runcard_loaded['features'] = ML_features
json_runcard_loaded['event_data'] = '/depot/cms/top/jpittard/Purdue_Analysis_EFT/pickel_files/sm_quad_pickles'
json_runcard_loaded['c_train'] = {'ctGRe': 2,'ctu8': 2, 'ctGRe_ctu8': 2, 'ctu8_ctGRe': 2}
json_runcard_loaded['epochs'] = 30
json_runcard_loaded['lr'] = 0.01
json_runcard_loaded['n_batches'] = 10  # Original: 50
json_runcard_loaded['patience'] = 15
json_runcard_loaded['n_dat'] = 1000000 # Original: 100000
json_runcard_loaded

In [ ]:
# 'output_dir', file location to save ML model
# "c_name", WC that the ML will find the ratio for against SM 

with open(runcard, 'w') as runcard_updated:
    json.dump(json_runcard_loaded, runcard_updated)

output_dir = './model/quad_limit_ctGRe_ctu8_mtt_y/quad_ctu8_ctGRe'
c_name = 'ctu8_ctGRe'

Training ML and getting Liklihood ratio

In [ ]:
# Runs ML

fitter = classifier.Fitter(json_path = runcard,
                           mc_run = 0,
                           c_name = c_name,
                           output_dir = output_dir,
                           print_log = True)

In [ ]:
#Gets ML model and calculates the liklihood ratio

path_to_models_root = os.path.join('./model/quad_limit_ctGRe_ctu8_mtt_y/quad_ctGRe_ctu8', time.strftime("2026/03/15"))
order = 'quad'

models_paths_dict = analyse.Analyse.build_path_dict(root=path_to_models_root,
                        order=order,
                        prefix='model')

analyser = analyse.Analyse(models_paths_dict, order, all=True)
analyser

In [ ]:
#Plots loss, only works if order = 'lin'

fig, _ = analyser.plot_loss_overview(c_name, order, xlim=5)

In [ ]:
# Edited theory ML4EFT calculations, so code can run
# Some might be irrelevant, need to check 

from __future__ import division
from ml4eft.core.truth import tt_prod
import numpy as np
import matplotlib
from matplotlib import pyplot as plt
from matplotlib import rc
from scipy.integrate import quad, dblquad
from scipy import integrate
import pylhe

mt = 0.17276
s = 14 ** 2
Gf = 0.000011663787
v = 1 / np.sqrt(Gf * np.sqrt(2)) * 10 ** -3
asQCD = 0.1179
LambdaSMEFT = 1
pb_convert = 3.894E2
yt = 1
try:
    import lhapdf

    p = lhapdf.mkPDF("NNPDF31_lo_as_0118", 0)
except ImportError:
    print("lhapdf not found: exact models will not be available")

class crossSectionSMEFT:
    def sigma_part_gg(self, hats, ctGRe, cut, order):
        if np.sqrt(hats) == 2 * mt:
            return 0

        sqrt = np.sqrt(1 - 4 * mt ** 2 / hats)
        kappa_11 = ((v ** 2 * yt ** 2 * asQCD) / (24 * LambdaSMEFT ** 4 * hats ** 3)) * (
                6 * np.sqrt(hats ** 5 * (hats - 4 * mt ** 2)) + hats * mt ** 2 * (
                -3 * np.sqrt(hats * (hats - 4 * mt ** 2)) - 8 * hats * np.log(1 - sqrt) + 8 * hats * np.log(sqrt + 1)))
        kappa_1 = (np.sqrt(np.pi) * v * yt * mt * asQCD) / (6 * LambdaSMEFT ** 2 * hats ** 2 * np.sqrt(2)) * (
                9 * np.sqrt(hats * asQCD * (hats - 4 * mt ** 2)) + 8 * hats * np.sqrt(asQCD) * (
                np.log(1 - np.sqrt(1 - 4 * mt ** 2 / hats)) - np.log(np.sqrt(1 - 4 * mt ** 2 / hats) + 1)))
        sm = (-np.pi * asQCD ** 2) / (12 * hats ** 3) * (
                4 * mt ** 4 * (np.log(1 - sqrt) - np.log(sqrt + 1)) + mt ** 2 * (
                31 * np.sqrt(hats * (hats - 4 * mt ** 2)) + 16 * hats * np.log(1 - sqrt) - 16 * hats * np.log(
            sqrt + 1)) + hats * (7 * np.sqrt(hats * (hats - 4 * mt ** 2)) + 4 * hats * np.log(
            1 - sqrt) - 4 * hats * np.log(sqrt + 1)))

        if order is None:
            return sm
        elif order == 'lin':
            return sm + ctGRe * kappa_1
        elif order == 'quad':
            return sm + ctGRe ** 2 * kappa_11
            
    def sigma_part_qq(self, hats, cuGRe, ctu8, order):
        if np.sqrt(hats)== 2 * mt:
            return 0

        sqrt = np.sqrt(1 - 4 * mt ** 2 / hats)
        kappa_11 = sqrt * (8 * np.pi * v ** 2 * yt ** 2 * asQCD * (8 * mt ** 2 + hats)) / (
                108 * np.pi * LambdaSMEFT ** 4 * hats)

        # kappa_22 = (2.0 / 9.0) * sqrt * (hats - mt ** 2) / (48 * np.pi * LambdaSMEFT ** 4)
        # kappa_22 = sqrt * (hats - mt ** 2) / (48 * np.pi * LambdaSMEFT ** 4)
        kappa_22 = (2.0 / 9.0) * (hats * sqrt - mt ** 2 * sqrt) / (12 * 4 * np.pi * LambdaSMEFT ** 4)

        kappa_1 = - (8 * np.sqrt(2 * np.pi) * v * mt * asQCD ** (3 / 2) * sqrt) / (9 * hats * LambdaSMEFT ** 2)
        kappa_2 = asQCD * sqrt * (2 * mt ** 2 + hats) / (27 * hats * LambdaSMEFT ** 2)
        sm = (8 * np.pi * asQCD ** 2 * (2 * mt ** 2 + hats) * sqrt) / (27 * hats ** 2)

        if order is None:
            return sm
        elif order == 'lin':
            return sm + cuGRe * kappa_1 + ctu8 * kappa_2
        elif order == 'quad':
            return sm + cuGRe ** 2 * kappa_11 + ctu8 ** 2 * kappa_22
xsec = crossSectionSMEFT()



def weight(sqrts, mu, x1, x2, c, order):
    ctGRe, cut = c
    hats = sqrts ** 2

    w_e = (xsec.sigma_part_gg(hats, ctGRe, cut, order)) * (p.xfxQ(21, x1, mu) * p.xfxQ(21, x2, mu))

    w_e += (xsec.sigma_part_qq(hats, ctGRe, 0, order)) * (
            p.xfxQ(1, x1, mu) * p.xfxQ(-1, x2, mu) +
            p.xfxQ(1, x2, mu) * p.xfxQ(-1, x1, mu) +
            p.xfxQ(3, x1, mu) * p.xfxQ(-3, x2, mu) +
            p.xfxQ(3, x2, mu) * p.xfxQ(-3, x1, mu) +
            p.xfxQ(5, x1, mu) * p.xfxQ(-5, x2, mu) +
            p.xfxQ(5, x2, mu) * p.xfxQ(-5, x1, mu)
    )

    w_e += (xsec.sigma_part_qq(hats, ctGRe, cut, order)) * (
            p.xfxQ(2, x1, mu) * p.xfxQ(-2, x2, mu) +
            p.xfxQ(2, x2, mu) * p.xfxQ(-2, x1, mu) +
            p.xfxQ(4, x1, mu) * p.xfxQ(-4, x2, mu) +
            p.xfxQ(4, x2, mu) * p.xfxQ(-4, x1, mu)
    )

    return w_e

v_weight = np.vectorize(weight, otypes=[np.float])
v_weight.excluded.add(4)

def new_dsigma_dmtt_dy(y, mtt, c=None, order=None):
    mtt = mtt/1000 # ML4EFT needs the values in TeV but using GeV data
    
    s = 14 ** 2

    if c is None:
        c = np.zeros(2)
    if mtt == 2 * mt: return 0  # if at threshold return zero
    if y < np.log(np.sqrt(s) / (mtt)):  # check whether x = {mtt, y} falls inside the physically allowed region
        x1 = mtt / np.sqrt(s) * np.exp(y)
        x2 = mtt / np.sqrt(s) * np.exp(-y)
        dsigma_dmtt_dy = 2 * mtt / s * v_weight(mtt, 91.188, x1, x2, c, order) / (x1 * x2)
        return pb_convert * dsigma_dmtt_dy
    else:
        return 0

def new_likelihood_ratio_truth(events, c, features, process, order=None):
    n_features = len(features)

    if process == 'tt':
        c = np.array([c['ctGRe'], c['ctu8']])
        if n_features == 1:
            dsigma_0 = [tt_prod.dsigma_dmtt(*x[features], c, order) for _, x in events.iterrows()]  # EFT
            dsigma_1 = [tt_prod.dsigma_dmtt(*x[features]) for _, x in
                        events.iterrows()]  # SM
        elif n_features == 2:
            dsigma_0 = [new_dsigma_dmtt_dy(*x[features], c, order) for _, x in
                        events.iterrows()]  # EFT
            dsigma_1 = [new_dsigma_dmtt_dy(*x[features]) for _, x in
                        events.iterrows()]  # SM
        elif n_features == 3:
            dsigma_0 = [tt_prod.dsigma_dmtt_dy_dpt(*x[features], c, order) for
                        index, x in
                        events.iterrows()]  # EFT
            dsigma_1 = [tt_prod.dsigma_dmtt_dy_dpt(*x[features]) for
                        index, x in
                        events.iterrows()]  # SM

    dsigma_0 = np.array(dsigma_0, dtype = float)
    dsigma_1 = np.array(dsigma_1, dtype = float)

    ratio = np.divide(dsigma_0, dsigma_1, out=np.zeros_like(dsigma_0), where=dsigma_1 != 0)

    return ratio.flatten()

def edited_likelihood_ratio_nn(c, df=None, epoch=-1):

    return analyser.likelihood_ratio_nn(c, df)

def new_point_by_point_comp_med(df, c, features, process, order, ax, text=None):

    r_nn = edited_likelihood_ratio_nn(c, df=df)
    tau_nn = np.log(r_nn)

    r_truth = new_likelihood_ratio_truth(df, c, features, process, order)
    tau_truth = np.log(r_truth)

    fig = plt.figure(figsize=(8, 8))
    x = np.linspace(-0.4, 1.7, 100)

    ax.scatter(tau_truth, np.median(tau_nn, axis=0), s=2, color='red')
    ax.plot(x, x, linestyle='dashed', color='grey')
    ax.set_xlabel(r'$\log r(x, c)^{\rm{Unbinned}\;\rm{exact}}$')
    ax.set_ylabel(r'$\log r(x, c)^{\rm{Unbinned}\;\rm{ML}}$')
    ax.set_xlim((np.min(x), np.max(x)))
    ax.set_ylim((np.min(x), np.max(x)))

    if text is not None:
        ax.text(0.95, 0.1, text,
                horizontalalignment='right',
                verticalalignment='center',
                transform=ax.transAxes)

    fig.tight_layout()

In [ ]:
# Comparison plot of a ML model, only works for mttbar and Y features

fig, ax = plt.subplots()

analyser.point_by_point_comp_med(df_sm.sample(n=1000, random_state=2112),
                        {'ctGRe': 2, 'ctu8': 0},
                        ['gen_ttbar_rapidity','gen_ttbar_mass'],
                        'tt', 'quad', ax)

Limit Plots

In [ ]:
# json for using ML4EFT limit calculations
# "Order", quad or lin
# "Process", ttparton
# with additional file called info.txt that has the WC value, i.e. "2"
# "Mode", how the limit is calculated: nn, binned, truth
# "Features", features used in theory calculations (only needed for binned and truth) must be mttbar and Y
# "nlive", does nothing, needed for pymultinest but im not using it
# "lumi", luminocity
# "bins", when mode is binned, selects the binns to scan over
# "path_to_models", location of the ML models, files used must have format: {path to dir}/model_{WC name 1}_{WC name 2}
# "path_to_theory_pred" files used must have format: {path to dir}/ttparton_{WC name 1}_{WC name 2}/events_0.pkl
# "Observed_data_path" i am using the SM data that is used in the theory path, this might not be the most correct
# "Results_path" not needed


optimizer_json = {
              "fit_id": "test_fit",
              "order": "quad",
              "process": "ttparton",
              "mode": "nn",
              "th_features": ['gen_ttbar_rapidity','gen_ttbar_mass'],
            
              "nlive": 1000,
              "lumi": 300, 

              "bins": {'gen_ttbar_rapidity':[-3.0,-1.5,0,1.5,3.0],
                       'gen_ttbar_mass': [1450,2500,np.inf],
              },
            
              "path_to_models": "/depot/cms/top/jpittard/Purdue_Analysis_EFT/EFT_param_classifier/model/quad_limit_ctGRe_ctu8_mtt_y/",
            
              "path_to_theory_pred": "/depot/cms/top/jpittard/Purdue_Analysis_EFT/pickel_files/limit_sm_quad_pickles",
            
              "observed_data_path": "/depot/cms/top/jpittard/Purdue_Analysis_EFT/pickel_files/ttparton_sm/events_0.pkl.gz",
            
              "results_path": "limit_results/"
            }

In [ ]:
# calculates the limit for the two WCs 

optimizer = optimize.Optimize(optimizer_json, coeff = ['ctGRe','ctu8'], rep = None)
optimizer

In [ ]:
# check if the data was loaded in as expected

optimizer.th_pred.th_dict

In [ ]:
#Calculates the limit space for the two WC
#spans over values for c1 and c2

c1_val = np.linspace(-1,1,80)
c2_val = np.linspace(-1,1,80)

liklihood = np.zeros((len(c1_val),len(c2_val)))

for i, c1 in enumerate(c1_val):
    for j, c2 in enumerate(c2_val):
        cube = np.array([c1,c2])

        liklihood[i,j] = optimizer.log_like_nn(cube)  # limit for NN
        #liklihood[i,j] = optimizer.log_like_binned(cube)  # limit for binned
        #liklihood[i,j] = optimizer.log_like_truth(cube)  # limit for theory
    print(f"done with i = {i} interval")

In [ ]:
# Plots the 68% and 95% confidence interval

liklihood_max = np.max(liklihood)

q_test = 2 * np.log(abs(liklihood / liklihood_max)) #test statistic
print(q_test)

sm = [0,0]
plt.contour(c1_val,c2_val,q_test.T, levels = [2.30,5.99])
plt.scatter(sm,sm,color = 'black', s = 4)
plt.xlabel("ctGRe")
plt.ylabel("ctu8")
plt.xlim(-1,1)
plt.ylim(-1,1)
plt.title("Confidence Intervals: 68%, 95%")
#plt.show()

#plt.figure()
plt.pcolormesh( c1_val, c2_val, q_test.T,)
plt.xlabel("ctGRe")
plt.ylabel("ctu8")
plt.title("Likelihood density")
plt.colorbar(label="$\Delta L$")
plt.show()